In [1]:
import sys, numpy, pandas, sklearn
import matplotlib

print("Python:      ", sys.version.split()[0])
print("numpy:       ", numpy.__version__)
print("pandas:      ", pandas.__version__)
print("matplotlib:  ", matplotlib.__version__)
print("scikit-learn:", sklearn.__version__)

Python:       3.12.3
numpy:        2.0.2
pandas:       2.3.3
matplotlib:   3.9.4
scikit-learn: 1.6.1


## Task 1 — Data Preprocessing

**Student ID seed:** last 4 digits = **3039**

In [2]:
#2
import pandas as pd 
raw = pd.read_csv("data/almaty_apartments_raw.csv")
ids = raw["listing_id"].drop_duplicates().sample(frac=0.9, random_state=3039)
df = raw[raw["listing_id"].isin(ids)].reset_index(drop=True)

print(df.shape)
print(df.dtypes)
print(df.isna().sum())
print(df.describe(include="all").T)

(1153, 11)
listing_id           object
listed_date          object
district             object
rooms                object
area_m2              object
floor                 int64
total_floors          int64
year_built          float64
building_type        object
ceiling_height_m    float64
price_kzt            object
dtype: object
listing_id            0
listed_date           0
district              0
rooms                 0
area_m2               0
floor                 0
total_floors          0
year_built           59
building_type         0
ceiling_height_m    141
price_kzt             0
dtype: int64
                   count unique         top freq        mean         std  min  \
listing_id          1153   1080   ALM-10932    3         NaN         NaN  NaN   
listed_date         1153    248  2026-04-18   12         NaN         NaN  NaN   
district            1153     48       Medeu  143         NaN         NaN  NaN   
rooms               1153      6           2  430         NaN      

In [3]:
for col in ["listed_date", "rooms", "area_m2", "price_kzt"]:
    print(f"=== {col} ===")
    print(df[col].sample(10).tolist())
    print()

=== listed_date ===
['2026-03-09', '2026-02-20', '2026-02-04', '2026-06-04', '2026-03-18', '2026-01-19', '2026-01-29', '2026-05-20', '2026-05-31', '2026-05-06']

=== rooms ===
['2', '4', '2', '1', '2', '1', '1', '1', '3', '3']

=== area_m2 ===
['38.8', '565.0', '43.1', '40,8', '67.9', '64.6', '102.1 m²', '66.4', '61.2', '54.8']

=== price_kzt ===
['26670000', '25380000', '25390000', '25660000', '89 740 000 ₸', '66270000', '31370000', '51150000', '75370000', '34650000']



In [4]:
print("Shape:", df.shape)
print("Exact duplicates:", df.duplicated().sum())
print("Shared listing_id:", df.drop_duplicates()["listing_id"].duplicated().sum())

Shape: (1153, 11)
Exact duplicates: 31
Shared listing_id: 42


In [5]:
reposts = df[df["listing_id"].duplicated(keep=False)].sort_values(["listing_id", "listed_date"])
print(reposts[["listing_id", "listed_date", "district", "area_m2", "price_kzt"]].head(10))

    listing_id listed_date   district area_m2  price_kzt
961  ALM-10003  2026-05-04     Auezov    48.6   28320000
7    ALM-10003  2026-05-29     Auezov    48.6   26120000
814  ALM-10003  2026-05-29     Auezov    48.6   26120000
933  ALM-10009  2025-12-03  Medeuskiy    33.4   42700000
178  ALM-10009  2026-01-19  Medeuskiy    33.4   39400000
466  ALM-10016  2026-01-20     Auezov    39.7   35130000
443  ALM-10016  2026-01-29     Auezov    39.7   32100000
245  ALM-10021  2026-04-06     Alatau    71.3   32300000
592  ALM-10021  2026-05-09     Alatau    71.3   30090000
48   ALM-10056  2026-02-08  Bostandyk   107.2  101060000


In [6]:
# Странные даты
bad_dates = df[~df["listed_date"].str.match(r"\d{4}-\d{2}-\d{2}", na=False)]
print("Странных дат:", len(bad_dates))
print(bad_dates["listed_date"].head().tolist())

# Странные rooms
bad_rooms = df[~df["rooms"].str.match(r"^\d+$", na=False)]
print("\nСтранных rooms:", len(bad_rooms))
print(bad_rooms["rooms"].value_counts())

# Проверка rooms
print("\nВсе значения rooms:")
print(df["rooms"].value_counts())

Странных дат: 0
[]

Странных rooms: 31
rooms
studio    31
Name: count, dtype: int64

Все значения rooms:
rooms
2         430
3         296
1         254
4         114
studio     31
5          28
Name: count, dtype: int64


### Question 1 — Mis-typed columns

Four columns have `object` dtype but should be numbers or dates:

| Column | Current | Should be | Reason |
|---|---|---|---|
| `listed_date` | object | datetime | Pandas does not parse dates automatically |
| `rooms` | object | int | 31 rows contain the text `"studio"` instead of a number |
| `area_m2` | object | float | Mixed formats: some values have `m²` suffix, some use comma as decimal separator |
| `price_kzt` | object | float | Mixed formats: some values have spaces as thousand separators, some have commas, and some have the `₸` symbol |

### Question 2 — Grain

- Exact duplicate rows: **31**
- Rows sharing a `listing_id` (after removing exact duplicates): **42**

**Grain of the cleaned table:** one row = one final listing posting — one flat, one price, one date.

Reposts differ in `listed_date` and `price_kzt` — all other columns are identical.

### Question 3 — Validation spec

| Column | Type | Valid range | Reasoning |
|---|---|---|---|
| listing_id | string | unique, non-empty | Each listing has a unique ID |
| listed_date | datetime | 2015-01-01 — 2026-12-31 | Listings started ~2015; no future dates |
| district | category | 8 canonical names | Almaty has exactly 8 districts |
| rooms | int | 1 — 10 | Studio = 1; large apartments up to ~10 rooms |
| area_m2 | float | 10 — 500 | Smallest studio ~10 m²; largest penthouse ~500 m² |
| floor | int | 1 — total_floors | Cannot be 0 or above the building |
| total_floors | int | 1 — 40 | Low-rise to high-rise in Almaty |
| year_built | int | 1900 — 2026 | Mass housing started ~1900; no future dates |
| building_type | category | panel / brick / monolith | Three construction types |
| ceiling_height_m | float | 2.0 — 5.0 | Old to modern apartments |
| price_kzt | float | 5_000_000 — 2_000_000_000 | Cheapest to most expensive |

## Step 2 — Load and inspect

**Seed:** last 4 digits of student ID = **3039**

### Q1 — Mis-typed columns

Four columns have `object` dtype but should be numbers or dates:

1. **listed_date** → should be datetime. Reason: pandas doesn't parse dates automatically.
2. **rooms** → should be int. Reason: 31 rows contain the text `"studio"` instead of a number.
3. **area_m2** → should be float. Reason: mixed formats — some have `m²` suffix, some use comma instead of dot.
4. **price_kzt** → should be float. Reason: mixed formats — some have spaces, commas, or `₸` symbol.

### Q2 — Grain

- Exact duplicates: **31**
- Rows sharing a `listing_id`: **42**

Grain: one row = one final listing posting (one flat, one price, one date).

Reposts differ in `listed_date` and `price_kzt` — everything else is identical.

### Q3 — Validation spec

- **listing_id** — string, unique
- **listed_date** — datetime, 2015–2026
- **district** — one of 8 canonical names
- **rooms** — int, 1–10
- **area_m2** — float, 10–350 (updated in Step 4)
- **floor** — int, 1 to total_floors
- **total_floors** — int, 1–40
- **year_built** — int, 1900–2026
- **building_type** — panel / brick / monolith
- **ceiling_height_m** — float, 2.0–5.0
- **price_kzt** — float, 5M — 2B

In [43]:
print("floor = -1:", (df["floor"] == -1).sum())
print("year_built = 0:", (df["year_built"] == 0).sum())
print("ceiling_height_m = 0:", (df["ceiling_height_m"] == 0).sum())

floor = -1: 0
year_built = 0: 0
ceiling_height_m = 0: 0


In [44]:
print("Before:", df.shape)
df = df.drop_duplicates()
print("After exact dedup:", df.shape)

Before: (1071, 13)
After exact dedup: (1071, 13)


In [45]:
reposts = df[df["listing_id"].duplicated(keep=False)].sort_values(["listing_id", "listed_date"])
print("Строк в перерепостах:", len(reposts))
print(reposts[["listing_id", "listed_date", "price_kzt"]].head(20))

Строк в перерепостах: 0
Empty DataFrame
Columns: [listing_id, listed_date, price_kzt]
Index: []


In [46]:
df["listed_date"] = pd.to_datetime(df["listed_date"])   # чтобы сортировка работала
df = df.sort_values("listed_date").drop_duplicates("listing_id", keep="last")
df = df.reset_index(drop=True)
print("After repost resolution:", df.shape)

After repost resolution: (1071, 13)


### 3a — Duplicates and reposts

I removed **31** exact duplicate rows and resolved **42** reposts (rows 
sharing a `listing_id` after exact deduplication).

**Decision:** keep the **most recent** posting per `listing_id`, using:
\`\`\`python
df.sort_values("listed_date").drop_duplicates("listing_id", keep="last")
\`\`\`

**Why the newest posting:**
A repost is a **deliberate action** by the seller. If the price had not 
changed, there would be no reason to publish the listing again. The fact 
that a new row exists means the previous information (price, date) became 
**outdated** — the seller reconsidered the price after seeing market 
reaction. Therefore the newest posting reflects the current asking price, 
which is what the model should learn from.

If I had kept the **oldest** posting instead, the training data would 
reflect prices that sellers themselves had already revised — i.e. 
information they no longer stand behind.

In [47]:
#3b
print(df["district"].value_counts())

district
Medeu        157
Auezov       148
Bostandyk    147
Almaly       130
Turksib      129
Zhetysu      125
Nauryzbay    124
Alatau       111
Name: count, dtype: int64


In [48]:
df["district_clean"] = df["district"].str.strip().str.casefold()
print(df["district_clean"].value_counts())

district_clean
medeu        157
auezov       148
bostandyk    147
almaly       130
turksib      129
zhetysu      125
nauryzbay    124
alatau       111
Name: count, dtype: int64


In [49]:
df["district_clean"] = df["district"].str.strip().str.casefold()

mapping = {

    "alatau": "Alatau", "almaly": "Almaly", "auezov": "Auezov",
    "bostandyk": "Bostandyk", "medeu": "Medeu", "nauryzbay": "Nauryzbay",
    "turksib": "Turksib", "zhetysu": "Zhetysu",

    "alatauskiy": "Alatau", "almalinskiy": "Almaly",
    "auezovskiy": "Auezov", "bostandykskiy": "Bostandyk",
    "medeuskiy": "Medeu", "nauryzbayskiy": "Nauryzbay",
    "turksibskiy": "Turksib", "zhetysuskiy": "Zhetysu",

    "алатауский": "Alatau", "алмалинский": "Almaly",
    "ауэзовский": "Auezov", "бостандыкский": "Bostandyk",
    "медеуский": "Medeu", "наурызбайский": "Nauryzbay",
    "турксибский": "Turksib", "жетысуский": "Zhetysu",
}

df["district"] = df["district_clean"].map(mapping)
df = df.drop(columns=["district_clean"])

print("Unique:", df["district"].nunique())
print("NaN:", df["district"].isna().sum())
print(df["district"].value_counts())

Unique: 8
NaN: 0
district
Medeu        157
Auezov       148
Bostandyk    147
Almaly       130
Turksib      129
Zhetysu      125
Nauryzbay    124
Alatau       111
Name: count, dtype: int64


In [50]:
#3c
df["rooms"] = df["rooms"].replace("studio", "1")
df["rooms"] = pd.to_numeric(df["rooms"], errors="raise")
print("rooms dtype:", df["rooms"].dtype)

rooms dtype: int64


In [51]:
print(df["area_m2"].sample(20).tolist())

[44.9, 79.2, 92.6, 25.8, 53.6, 55.5, 57.2, 72.4, 111.7, 41.6, 73.9, 83.0, 36.1, 61.7, 56.3, 41.7, 72.2, 43.9, 80.1, 54.6]


In [52]:
df["area_m2"] = df["area_m2"].str.replace("m²", "", regex=False).str.strip()
df["area_m2"] = df["area_m2"].str.replace(",", ".", regex=False)
df["area_m2"] = pd.to_numeric(df["area_m2"], errors="coerce")
print("NaN after:", df["area_m2"].isna().sum())

AttributeError: Can only use .str accessor with string values!

In [ ]:
print(df["price_kzt"].sample(20).tolist())

['148590000', '54240000', '36650000', '51980000', '48620000', '42410000', '73860000', '43220000', '55940000', '28,870,000', '27030000', '24120000', '29050000', '33610000', '27100000', '41240000', '44570000', '23610000', '98420000', '32610000']


In [ ]:

df["price_kzt"] = df["price_kzt"].str.replace("₸", "", regex=False)
df["price_kzt"] = df["price_kzt"].str.replace(" ", "", regex=False)
df["price_kzt"] = df["price_kzt"].str.replace("\u00a0", "", regex=False)
df["price_kzt"] = df["price_kzt"].str.replace(",", "", regex=False)
df["price_kzt"] = pd.to_numeric(df["price_kzt"], errors="coerce")

print("price_kzt dtype:", df["price_kzt"].dtype)
print("NaN after:", df["price_kzt"].isna().sum())

price_kzt dtype: int64
NaN after: 0


In [ ]:
print(df.dtypes)

listing_id                  object
listed_date         datetime64[ns]
district                    object
rooms                        int64
area_m2                    float64
floor                        int64
total_floors                 int64
year_built                 float64
building_type               object
ceiling_height_m           float64
price_kzt                    int64
dtype: object


In [ ]:
#4.1
import numpy as np
df["floor"] = df["floor"].replace(-1, np.nan)
df["year_built"] = df["year_built"].replace(0, np.nan)
df["ceiling_height_m"] = df["ceiling_height_m"].replace(0, np.nan)

print("floor NaN:", df["floor"].isna().sum())
print("year_built NaN:", df["year_built"].isna().sum())
print("ceilling_height_m NaN:", df["ceiling_height_m"].isna().sum())

floor NaN: 10
year_built NaN: 81
ceilling_height_m NaN: 151


In [ ]:
#4.2
impossible_floor = df[df["floor"] > df["total_floors"]]
print("Этаж выше дома:", len(impossible_floor))
print(impossible_floor[["listing_id", "floor", "total_floors"]].head(10))

Этаж выше дома: 11
    listing_id  floor  total_floors
6    ALM-10729   15.0            12
118  ALM-10851   13.0            12
148  ALM-10944    4.0             3
308  ALM-10044   30.0            25
609  ALM-10636    6.0             5
700  ALM-10628    6.0             5
748  ALM-10321   10.0             5
823  ALM-10266   13.0            12
877  ALM-10881    8.0             5
985  ALM-10074   11.0             9


In [ ]:
# Остальные проверки по validation spec
print("rooms < 1:        ", (df["rooms"] < 1).sum())
print("area_m2 < 10:     ", (df["area_m2"] < 10).sum())
print("area_m2 > 500:    ", (df["area_m2"] > 500).sum())
print("price_kzt <= 0:   ", (df["price_kzt"] <= 0).sum())
print("year_built < 1900:", (df["year_built"] < 1900).sum())
print("year_built > 2026:", (df["year_built"] > 2026).sum())
print("ceiling_h < 2.0:  ", (df["ceiling_height_m"] < 2.0).sum())
print("ceiling_h > 5.0:  ", (df["ceiling_height_m"] > 5.0).sum())

rooms < 1:         0
area_m2 < 10:      0
area_m2 > 500:     6
price_kzt <= 0:    0
year_built < 1900: 0
year_built > 2026: 0
ceiling_h < 2.0:   0
ceiling_h > 5.0:   0


In [ ]:
#4.3
df["price_per_m2"] = df["price_kzt"] / df["area_m2"]

df["area_before_fix"] = df["area_m2"]

print("Добавлены price_per_m2 и area_before_fix")
print(df[["price_kzt", "area_m2", "price_per_m2"]].head())

Добавлены price_per_m2 и area_before_fix
   price_kzt  area_m2  price_per_m2
0   63800000     77.4  8.242894e+05
1   28410000     65.9  4.311077e+05
2   55950000     49.7  1.125755e+06
3   52620000     50.5  1.041980e+06
4   40720000     39.9  1.020551e+06


In [ ]:
q1 = df["price_per_m2"].quantile(0.25)
q3 = df["price_per_m2"].quantile(0.75)
iqr = q3 - q1
low = q1 - 1.5 * iqr
high = q3 + 1.5 * iqr

outside_global = (df["price_per_m2"] < low) | (df["price_per_m2"] > high)

print(f"Q1 = {q1:,.0f}")
print(f"Q3 = {q3:,.0f}")
print(f"IQR = {iqr:,.0f}")
print(f"Low fence:  {low:,.0f}")
print(f"High fence: {high:,.0f}")
print(f"Строк outside (global): {outside_global.sum()}")

Q1 = 522,460
Q3 = 918,748
IQR = 396,288
Low fence:  -71,972
High fence: 1,513,180
Строк outside (global): 0


In [ ]:
grouped_q1 = df.groupby("district")["price_per_m2"].quantile(0.25)
grouped_q3 = df.groupby("district")["price_per_m2"].quantile(0.75)
grouped_iqr = grouped_q3 - grouped_q1

low_g = df["district"].map(grouped_q1 - 1.5 * grouped_iqr)
high_g = df["district"].map(grouped_q3 + 1.5 * grouped_iqr)

outside_group = (df["price_per_m2"] < low_g) | (df["price_per_m2"] > high_g)

print(f"Строк outside (by district): {outside_group.sum()}")

Строк outside (by district): 10


In [ ]:
#4.4
flagged = df[outside_group].sort_values("price_per_m2")
print(f"Всего flagged (by district): {len(flagged)}")
print()
print(flagged[["listing_id", "district", "area_m2", "price_kzt", "price_per_m2"]].head(20))

Всего flagged (by district): 10

    listing_id   district  area_m2  price_kzt   price_per_m2
175  ALM-10405    Turksib    356.0   18830000   52893.258427
939  ALM-10440     Auezov    879.0   48370000   55028.441411
990  ALM-10069    Zhetysu    481.0   27710000   57609.147609
756  ALM-10909     Alatau    565.0   32610000   57716.814159
506  ALM-10921    Zhetysu    668.0   38560000   57724.550898
207  ALM-11122    Zhetysu    647.0   40670000   62859.350850
652  ALM-10264  Bostandyk    495.0   42680000   86222.222222
317  ALM-10058  Bostandyk    555.0   66270000  119405.405405
245  ALM-10072      Medeu    530.0   64270000  121264.150943
828  ALM-10605  Nauryzbay     54.3   36960000  680662.983425


In [ ]:
# Смотрим, что удаляем — area > 500 м²
to_drop = df[df["area_m2"] > 500]
print("Строк к удалению:", len(to_drop))
print(to_drop[["listing_id", "district", "area_m2", "price_kzt"]].to_string())
print()

# Удаляем
print("Before:", df.shape)
df = df[df["area_m2"] <= 500].reset_index(drop=True)
print("After: ", df.shape)

Строк к удалению: 6
    listing_id   district  area_m2  price_kzt
207  ALM-11122    Zhetysu    647.0   40670000
245  ALM-10072      Medeu    530.0   64270000
317  ALM-10058  Bostandyk    555.0   66270000
506  ALM-10921    Zhetysu    668.0   38560000
756  ALM-10909     Alatau    565.0   32610000
939  ALM-10440     Auezov    879.0   48370000

Before: (1080, 13)
After:  (1074, 13)


In [ ]:
print("Before:", df.shape)
df = df[df["area_m2"] <= 350].reset_index(drop=True)
print("After: ", df.shape)

Before: (1074, 13)
After:  (1071, 13)


In [ ]:
import numpy as np

bad_floor = df["floor"] > df["total_floors"]
print("Fixing rows:", bad_floor.sum())
df.loc[bad_floor, "floor"] = np.nan
print("Floor NaN now:", df["floor"].isna().sum())

Fixing rows: 11
Floor NaN now: 21


In [ ]:
print("=== After Step 4 ===")
print("Shape:", df.shape)
print()
print("Sentinels:")
print("  floor = -1:         ", (df["floor"] == -1).sum())
print("  year_built = 0:     ", (df["year_built"] == 0).sum())
print("  ceiling_height_m = 0:", (df["ceiling_height_m"] == 0).sum())
print()
print("Impossible:")
print("  floor > total_floors:", (df["floor"] > df["total_floors"]).sum())
print("  area_m2 > 350:      ", (df["area_m2"] > 350).sum())

=== After Step 4 ===
Shape: (1071, 13)

Sentinels:
  floor = -1:          0
  year_built = 0:      0
  ceiling_height_m = 0: 0

Impossible:
  floor > total_floors: 0
  area_m2 > 350:       0
